[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shashvindu/gonu-jha-ai-master-jee/blob/main/NLP/nlp_04_stemming_vs_lemmatization.ipynb) - run this notebook in your browser, nothing to install.

# NLP #4: Stemming vs Lemmatization

**Gonu Jha AI Master Jee** · Interview prep notebook for the video *Stemming vs Lemmatization* (taught by Gonu Jha Ji with Priya, Rahul and Sneha).

Run the cells from top to bottom (`Shift + Enter`). Every output you see was produced by running this notebook. Change the examples and run them again - that is the best way to learn.

## What you will learn
1. Why we normalize word forms (run, runs, running, ran)
2. Stemming with Porter, Snowball and Lancaster - and their mistakes
3. Lemmatization with WordNet - and why it needs a POS tag
4. **Automatic lemmatization of a full sentence** with a POS tagger
5. Over-stemming vs under-stemming, speed comparison, which one to use when

In [1]:
# One-time setup (uncomment if a library is missing):
# !pip install nltk scikit-learn pandas
import nltk
for pkg in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    nltk.download(pkg, quiet=True)
print("Ready!")

Ready!


## 1. Key definitions

| Term | Meaning | Example |
|---|---|---|
| **Normalization** | Reducing different forms of a word to one base form | running, runs → run |
| **Stemming** | Chopping suffixes with fixed rules (fast, may give non-words) | studies → *studi* |
| **Stem** | The output of a stemmer | *univers* |
| **Lemmatization** | Dictionary lookup + part of speech → the real base word | studies → *study*, better → *good* |
| **Lemma** | The dictionary form of a word | *run* is the lemma of *ran* |
| **POS** (part of speech) | Noun, verb, adjective, adverb... | *running* (verb) vs *running* (noun) |

> **Gonu Jha's story:** the fast cook chops everything (stemming); the careful cook checks the recipe book first (lemmatization).

## 2. Stemming: three algorithms side by side

In [2]:
import pandas as pd
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

words = ["running", "runs", "ran", "studies", "studying", "better", "university", "universe",
         "happiness", "generously", "flies", "organization"]
porter, snowball, lancaster = PorterStemmer(), SnowballStemmer("english"), LancasterStemmer()
pd.DataFrame({"word": words,
              "Porter": [porter.stem(w) for w in words],
              "Snowball": [snowball.stem(w) for w in words],
              "Lancaster": [lancaster.stem(w) for w in words]})

,word,Porter,Snowball,Lancaster
0,running,run,run,run
1,runs,run,run,run
2,ran,ran,ran,ran
3,studies,studi,studi,study
4,studying,studi,studi,study
5,better,better,better,bet
6,university,univers,univers,univers
7,universe,univers,univers,univers
8,happiness,happi,happi,happy
9,generously,gener,generous,gen


Observe:
- **Porter** (1980) is the classic. *studies* → *studi* (not a word), *university* and *universe* → *univers* (over-stemming).
- **Snowball** (Porter2) is a cleaner, improved Porter - the usual default.
- **Lancaster** is the most aggressive: short stems, more over-stemming.
- Nobody turns *ran* into *run*: suffix rules cannot handle irregular verbs.

## 3. Lemmatization with WordNet

In [3]:
from nltk.stem import WordNetLemmatizer
wnl = WordNetLemmatizer()
video_words = ["running", "runs", "ran", "studies", "studying", "better", "university", "universe"]
print("default (noun):", [wnl.lemmatize(w) for w in video_words])
print("as verbs      :", [wnl.lemmatize(w, pos="v") for w in video_words])
print("better as adj :", wnl.lemmatize("better", pos="a"))

default (noun): ['running', 'run', 'ran', 'study', 'studying', 'better', 'university', 'universe']
as verbs      : ['run', 'run', 'run', 'study', 'study', 'better', 'university', 'universe']
better as adj : good


Without a POS tag, WordNet treats every word as a **noun** - so *running* stays *running* and *ran* stays *ran*. The POS tag is the key.

## 4. Lemmatizing a real sentence automatically
In practice nobody types the POS by hand: run a POS tagger first, map its tags to WordNet tags, then lemmatize.

In [4]:
from nltk import pos_tag, word_tokenize
from nltk.corpus import wordnet

def wordnet_pos(tag):
    """Penn Treebank tag (from pos_tag) -> WordNet POS."""
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

sentence = "The children were running and the better players studied the rules carefully"
tagged = pos_tag(word_tokenize(sentence))
rows = [(w, t, wnl.lemmatize(w.lower(), wordnet_pos(t)), porter.stem(w)) for w, t in tagged]
pd.DataFrame(rows, columns=["word", "POS tag", "lemma", "Porter stem"])

,word,POS tag,lemma,Porter stem
0,The,DT,the,the
1,children,NNS,child,children
2,were,VBD,be,were
3,running,VBG,run,run
4,and,CC,and,and
5,the,DT,the,the
6,better,JJR,good,better
7,players,NNS,player,player
8,studied,VBD,study,studi
9,the,DT,the,the


*children → child, were → be, running → run, studied → study, better → good* - only possible with the dictionary + POS. (spaCy does all of this automatically with `token.lemma_`.)

## 5. Over-stemming vs under-stemming

| | What happens | Example | Hurts |
|---|---|---|---|
| **Over-stemming** | Different meanings get the same stem | university, universe → *univers* | **Precision** (wrong documents match) |
| **Under-stemming** | Same meaning gets different stems | ran → *ran*, running → *run* | **Recall** (relevant documents are missed) |

In [5]:
pairs = [("university", "universe"), ("organization", "organ"), ("news", "new"), ("ran", "running"), ("data", "datum")]
for a, b in pairs:
    same = porter.stem(a) == porter.stem(b)
    print(f"{a:13} -> {porter.stem(a):10} | {b:10} -> {porter.stem(b):8} | same stem: {same}")

university    -> univers    | universe   -> univers  | same stem: True
organization  -> organ      | organ      -> organ    | same stem: True
news          -> news       | new        -> new      | same stem: False
ran           -> ran        | running    -> run      | same stem: False
data          -> data       | datum      -> datum    | same stem: False


## 6. Speed: why search engines still use stemming
A fair comparison: on real text, a stemmer needs only the words, but a lemmatizer first needs a **POS tagger**
for every sentence (section 4). Time both full pipelines on the same 300 sentences:

In [6]:
import time
sentences = [sentence] * 300
t = time.perf_counter()
[[porter.stem(w) for w in word_tokenize(s)] for s in sentences]
stem_s = time.perf_counter() - t
t = time.perf_counter()
[[wnl.lemmatize(w.lower(), wordnet_pos(p)) for w, p in pos_tag(word_tokenize(s))] for s in sentences]
lem_s = time.perf_counter() - t
print(f"tokenize + stem              : {stem_s:.2f} s")
print(f"tokenize + POS tag + lemmatize: {lem_s:.2f} s  ({lem_s / stem_s:.1f}x slower)")
print("(exact numbers depend on your machine; on billions of documents every millisecond counts)")

tokenize + stem              : 0.12 s
tokenize + POS tag + lemmatize: 0.19 s  (1.5x slower)
(exact numbers depend on your machine; on billions of documents every millisecond counts)


## 7. Which one when?
| Use case | Choice | Why |
|---|---|---|
| Search engines, large-scale retrieval | Stemming | Speed and recall |
| Chatbots, QA, text analytics | Lemmatization | Real, readable words |
| BERT / GPT | Neither | The subword tokenizer already handles word forms |
| Hindi / Indian languages | IndicNLP, Stanza, rule-based stemmers | English tools do not work |

## 8. Interview questions (from the video)

**Q1. Difference between stemming and lemmatization?**
Stemming = rule-based suffix chopping: fast, may give non-words. Lemmatization = dictionary + POS: slower,
gives real words. Give two examples: *studies → studi (stem) vs study (lemma)*, *better → better (stem) vs good (lemma)*.

**Q2. Why does lemmatization need a POS tag?**
The same word has different lemmas for different POS: *running* (noun) stays *running*, *running* (verb) → *run*;
*better* (adjective) → *good*. In practice: POS tagger first, then lemmatize (spaCy does it automatically).

**Q3. What are over-stemming and under-stemming?**
Over: unrelated words merge (university, universe) → hurts precision. Under: related words stay apart
(ran, run) → hurts recall.

**Q4. Would you stem text before feeding it to BERT?**
No (trick question). BERT was pre-trained on raw text, its WordPiece tokenizer handles word forms, and
non-words like *studi* hurt accuracy. Use stemming / lemmatization for BoW, TF-IDF and search instead.

### Common mistakes
- Calling `lemmatize()` without POS and expecting *ran → run*
- Saying lemmatization is always better (it is slower)
- Stemming text for transformer models
- Not mentioning over-stemming when asked about stemming

## 9. Practice
**Task (from the video):** what are the stem and lemma of *generously*, *happiness* and *flies*? Guess first!

In [7]:
# Solution
for w, pos in [("generously", "r"), ("happiness", "n"), ("flies", "v"), ("flies", "n")]:
    print(f"{w:11} pos={pos}  stem={porter.stem(w):9} lemma={wnl.lemmatize(w, pos=pos)}")

generously  pos=r  stem=gener     lemma=generously
happiness   pos=n  stem=happi     lemma=happiness
flies       pos=v  stem=fli       lemma=fly
flies       pos=n  stem=fli       lemma=fly


The surprise: the Porter stem of *flies* is *fli* (not a word), while the lemma is *fly* - as a verb (*he flies*) and also as a noun (*two flies*).

**Next video:** NLP #5 - Bag of Words and N-grams.